In [ ]:
%py
spark.catalog.setCurrentCatalog("purgo_databricks")

# PySpark script: Forecast ERP logic for f_inv_movmnt report generation per inv_txn_mapping.xlsx
# Purpose: Generate forecast report rows for f_inv_movmnt, mapping and validating fields as per inv_txn_mapping.xlsx
# Author: Giang Nguyen
# Date: 2025-09-29
# Description: This script implements the PySpark logic to generate a forecast report for f_inv_movmnt, mapping fields from f_order and related sources per inv_txn_mapping.xlsx. It enforces data quality rules, handles NULLs, validates types and formats, and outputs the required columns: txn_id, allocated_qty, delivery_dt, sched_dt, flag_key. All logic is implemented using DataFrame APIs, with strict Databricks conventions and error handling.

# Import required PySpark modules
from pyspark.sql import functions as F  
from pyspark.sql.types import DecimalType, BooleanType, StringType, StructType, StructField  

# -- CTE: Calculate allocated_qty per order and line number from f_order (sum of qty_1..qty_4, NULLs as 0.00)
def get_allocated_qty_df(f_order_df):
    """
    Calculate allocated_qty per order and line number from f_order.
    Args:
        f_order_df (DataFrame): Input DataFrame for f_order table.
    Returns:
        DataFrame: DataFrame with columns [order_nbr, order_line_nbr, delivery_dt, sched_dt, allocated_qty].
    """
    # If qty_1..qty_4 do not exist, fallback to order_qty only
    qty_cols = [c for c in ['qty_1', 'qty_2', 'qty_3', 'qty_4'] if c in f_order_df.columns]
    if qty_cols:
        allocated_qty_expr = sum([F.coalesce(F.col(c), F.lit(0.00)) for c in qty_cols])
    else:
        allocated_qty_expr = F.coalesce(F.col('order_qty'), F.lit(0.00))
    return (
        f_order_df
        .select(
            F.col('order_nbr'),
            F.col('order_line_nbr'),
            F.col('delivery_dt'),
            F.col('sched_dt'),
            allocated_qty_expr.cast(DecimalType(38,2)).alias('allocated_qty')
        )
    )

# -- CTE: Prepare f_order fields for join (delivery_dt, sched_dt)
def get_order_fields_df(f_order_df):
    """
    Prepare f_order fields for join (delivery_dt, sched_dt).
    Args:
        f_order_df (DataFrame): Input DataFrame for f_order table.
    Returns:
        DataFrame: DataFrame with columns [order_nbr, order_line_nbr, delivery_dt, sched_dt].
    """
    # delivery_dt: If not present, use current date in yyyymmdd as decimal(38,0)
    delivery_dt_col = F.when(
        F.col('delivery_dt').isNotNull() & (F.length(F.col('delivery_dt').cast(StringType())) == 8),
        F.col('delivery_dt')
    ).otherwise(F.date_format(F.current_date(), 'yyyyMMdd').cast(DecimalType(38,0)))
    # sched_dt: If not present, NULL
    sched_dt_col = F.when(
        F.col('sched_dt').isNotNull() & (F.length(F.col('sched_dt').cast(StringType())) == 8),
        F.col('sched_dt')
    ).otherwise(F.lit(None).cast(DecimalType(38,0)))
    return (
        f_order_df
        .select(
            F.col('order_nbr'),
            F.col('order_line_nbr'),
            delivery_dt_col.alias('delivery_dt'),
            sched_dt_col.alias('sched_dt')
        )
    )

# -- CTE: Main join of f_inv_movmnt to f_order (left outer join), mapping all required fields
def get_forecast_erp_df(f_inv_movmnt_df, allocated_qty_df, order_fields_df):
    """
    Main join of f_inv_movmnt to f_order (left outer join), mapping all required fields.
    Args:
        f_inv_movmnt_df (DataFrame): Input DataFrame for f_inv_movmnt table.
        allocated_qty_df (DataFrame): DataFrame with allocated_qty per order.
        order_fields_df (DataFrame): DataFrame with delivery_dt and sched_dt per order.
    Returns:
        DataFrame: DataFrame with columns [txn_id, allocated_qty, delivery_dt, sched_dt, flag_key].
    """
    # Join keys: order_nbr, order_line_nbr
    # If f_inv_movmnt does not have order_nbr/order_line_nbr, skip join and set allocated_qty=0.00, delivery_dt=current, sched_dt=None
    join_keys = ['order_nbr', 'order_line_nbr']
    missing_keys = [k for k in join_keys if k not in f_inv_movmnt_df.columns]
    if missing_keys:
        # No join possible, fallback
        result_df = (
            f_inv_movmnt_df
            .select(
                F.col('txn_id'),
                F.lit(0.00).cast(DecimalType(38,2)).alias('allocated_qty'),
                F.date_format(F.current_date(), 'yyyyMMdd').cast(DecimalType(38,0)).alias('delivery_dt'),
                F.lit(None).cast(DecimalType(38,0)).alias('sched_dt'),
                F.when(F.col('txn_id').isNotNull() & (F.length(F.trim(F.col('txn_id'))) > 0), F.lit(True)).otherwise(F.lit(False)).cast(BooleanType()).alias('flag_key')
            )
        )
    else:
        # Join allocated_qty
        m_df = f_inv_movmnt_df
        a_df = allocated_qty_df
        o_df = order_fields_df
        # To avoid duplicate columns, rename in allocated_qty_df and order_fields_df
        a_df = a_df.withColumnRenamed('delivery_dt', 'a_delivery_dt').withColumnRenamed('sched_dt', 'a_sched_dt')
        o_df = o_df.withColumnRenamed('delivery_dt', 'o_delivery_dt').withColumnRenamed('sched_dt', 'o_sched_dt')
        # Join allocated_qty
        joined_df = (
            m_df
            .join(a_df, on=['order_nbr', 'order_line_nbr'], how='left')
            .join(o_df, on=['order_nbr', 'order_line_nbr'], how='left')
        )
        # delivery_dt: from order_fields, else current date yyyymmdd
        delivery_dt_col = F.when(
            F.col('o_delivery_dt').isNotNull() & (F.length(F.col('o_delivery_dt').cast(StringType())) == 8),
            F.col('o_delivery_dt')
        ).otherwise(F.date_format(F.current_date(), 'yyyyMMdd').cast(DecimalType(38,0)))
        # sched_dt: from order_fields, else NULL
        sched_dt_col = F.when(
            F.col('o_sched_dt').isNotNull() & (F.length(F.col('o_sched_dt').cast(StringType())) == 8),
            F.col('o_sched_dt')
        ).otherwise(F.lit(None).cast(DecimalType(38,0)))
        result_df = (
            joined_df
            .select(
                F.col('txn_id'),
                F.coalesce(F.col('allocated_qty'), F.lit(0.00)).cast(DecimalType(38,2)).alias('allocated_qty'),
                delivery_dt_col.alias('delivery_dt'),
                sched_dt_col.alias('sched_dt'),
                F.when(F.col('txn_id').isNotNull() & (F.length(F.trim(F.col('txn_id'))) > 0), F.lit(True)).otherwise(F.lit(False)).cast(BooleanType()).alias('flag_key')
            )
        )
    return result_df

# -- Final SELECT: Validate and output required columns, enforce data quality rules
def get_validated_forecast_erp_df(forecast_erp_df):
    """
    Validate and output required columns, enforce data quality rules.
    Args:
        forecast_erp_df (DataFrame): Input DataFrame with forecast ERP columns.
    Returns:
        DataFrame: DataFrame with validated columns [txn_id, allocated_qty, delivery_dt, sched_dt, flag_key].
    """
    return (
        forecast_erp_df
        .filter(
            (F.col('allocated_qty').isNotNull()) &
            ((F.col('delivery_dt').isNull()) | (F.length(F.col('delivery_dt').cast(StringType())) == 8)) &
            ((F.col('sched_dt').isNull()) | (F.length(F.col('sched_dt').cast(StringType())) == 8))
        )
        .select(
            F.col('txn_id'),
            F.col('allocated_qty'),
            F.col('delivery_dt'),
            F.col('sched_dt'),
            F.col('flag_key')
        )
    )

# -- Main execution block
try:
    # Read source tables from Unity Catalog
    f_order_df = spark.table("purgo_databricks.purgo_playground.f_order")
    f_inv_movmnt_df = spark.table("purgo_databricks.purgo_playground.f_inv_movmnt")
    # Calculate allocated_qty CTE
    allocated_qty_df = get_allocated_qty_df(f_order_df)
    # Prepare order fields CTE
    order_fields_df = get_order_fields_df(f_order_df)
    # Main join and mapping
    forecast_erp_df = get_forecast_erp_df(f_inv_movmnt_df, allocated_qty_df, order_fields_df)
    # Final validation and output
    validated_forecast_erp_df = get_validated_forecast_erp_df(forecast_erp_df)
    # Persist to a working table (overwrite)
    validated_forecast_erp_df.write.format("delta").mode("overwrite").saveAsTable("purgo_databricks.purgo_playground.forecast_erp_report")
except Exception as e:
    # Handle errors gracefully
    print(f"Error occurred during forecast ERP report generation: {e}")
